In [1]:
import numpy as np
import pandas as pd
import concurrent.futures
import timeit
from functools import partial
from copy import deepcopy
import sys
from IPython.display import display
import os
import pprint
# Add measurement mcts python package to path
sys.path.append('../src/measurement_mcts')
from measurement_mcts.mcts.mcts import mcts_with_rollout
from measurement_mcts.mcts.tree_viz import render_pyvis
from measurement_mcts.state_evaluation.hertg import HERTG
from measurement_mcts.environment.measurement_control_env import MeasurementControlEnvironment
from measurement_mcts.utils.metrics import get_percent_done, save_environment_config

# Create the environment
env = MeasurementControlEnvironment(init_reset=False)

/home/austin/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Toy Measurement Control Initialized


In [2]:
# Save the object configurations to a folder for use in the experiment
num_trials = 100
folder = "trial_configs1"

# Create the folder if it doesn't exist
if not os.path.exists(folder):
    os.makedirs(folder)

for i in range(num_trials):
    env.reset() # Reset the environment to a new random state
    env.save_state(folder, f"trial_{i}") # Save the environment state to a file
    
# Save the environment configuration
save_environment_config(env, folder)

Environment configuration saved to trial_configs1/env_config.txt


In [5]:
def get_mcts_metrics(env, state, max_actions=200, LI=100,
                     EF=0.1, DF=1.0, HL=6, rollout_method='random_same',
                     hertg_method='static', rollout_pre_collision_stop=True) -> dict:
    """
    Run MCTS and return the metrics.
    params:
        env: the environment object
        state: the initial state of the environment
        max_actions: the maximum number of actions to take
        LI: the length of the interval for the HERTG method
        EF: the exploration factor for the HERTG method
        DF: the discount factor for the HERTG method
        rollout_method: the rollout method to use
        hertg_method: the HERTG method to use
        
    returns:
        metrics: a dictionary of metrics
    """
    
    # Change environment parameters and create HERTG object
    env.horizon_length = HL
    hertg = HERTG(state, env, method=hertg_method)
    
    # Create metric trackers
    cumulative_reward = 0.
    num_actions = 0
    done = False
    start_time = timeit.default_timer()
    for i in range(max_actions):
        # Run MCTS and take the best action
        root = mcts_with_rollout(env, state, LI, EF, DF, rollout_method,
                                 rollout_pre_collision_stop, hertg=hertg)
        best_action_idx = np.argmax(root.child_Q())
        state, reward, done = env.step(state, env.action_space[best_action_idx])
        
        # Reset the horizon to 0
        state_list = list(state)
        state_list[3] = 0
        state = tuple(state_list)
        
        # Increment the cumulative reward and number of actions
        cumulative_reward += reward
        num_actions = i + 1
        if done:
            break
        
    comp_time = timeit.default_timer() - start_time
    percent_done = get_percent_done(state, env)
    
    metrics = {
        'LI': LI,
        'EF': EF,
        'DF': DF,
        'HL': HL,
        'rollout_method': rollout_method,
        'hertg_method': hertg_method,
        'done': done,
        'percent_done': percent_done,
        'cumulative_reward': cumulative_reward,
        'num_actions': num_actions,
        'computation_time': comp_time,
        'computation_per_action': comp_time / num_actions,
    }
    
    return metrics

In [ ]:
def worker_wrapper(trial_config_name, rollout_method,
                   trial_config_path,
                   max_actions=200,
                   LI=100,
                   EF=0.1,
                   DF=1.0,
                   HL=6,
                   hertg_method='static',
                   rollout_pre_collision_stop=True):
    """
    Worker function that:
      - Instantiates a fresh environment.
      - Loads a pre-saved trial configuration from file.
      - Retrieves the state and object true state.
      - Runs get_mcts_metrics using the given rollout method and other parameters.
    
    Parameters:
        trial_config_name (str): Name of the trial config file (without the .pkl extension)
        rollout_method (str): Rollout method to use.
        trial_config_path (str): Directory where trial configuration files are stored.
        max_actions, LI, EF, DF, hertg_method, rollout_pre_collision_stop:
            Additional parameters passed to get_mcts_metrics.
    
    Returns:
        dict: Metrics dictionary from get_mcts_metrics.
    """
    # Create a new environment instance.
    env = MeasurementControlEnvironment()
    
    # Load the saved state and object configuration.
    # This call uses your custom load_state method.
    env.load_state(trial_config_path, trial_config_name)
    
    # Retrieve the state and true object state.
    state = env.get_state()
    
    # Run the MCTS metrics collection using the loaded configuration.
    metrics = get_mcts_metrics(
        env,
        state,
        max_actions=max_actions,
        LI=LI,
        EF=EF,
        DF=DF,
        HL=HL,
        rollout_method=rollout_method,
        hertg_method=hertg_method,
        rollout_pre_collision_stop=rollout_pre_collision_stop
    )
    
    # Optionally, record which trial configuration was used.
    metrics['trial_config'] = trial_config_name
    return metrics

def run_experiments_for_rollout_methods(rollout_methods, trial_config_path, trial_config_names, **kwargs):
    """
    For each rollout method, run experiments for every pre-saved trial configuration.
    
    Parameters:
        rollout_methods (list of str): List of rollout methods to test.
        trial_config_path (str): Directory containing the trial configuration files.
        trial_config_names (list of str): List of trial configuration file names (without the .pkl extension).
        kwargs: Additional keyword arguments to pass to the worker_wrapper.
    
    Returns:
        List[dict]: A list of dictionaries with the metrics for each run.
    """
    all_results = []
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = []
        # Schedule a job for each combination of rollout method and trial configuration.
        for method in rollout_methods:
            for config_name in trial_config_names:
                futures.append(
                    executor.submit(
                        worker_wrapper,
                        trial_config_name=config_name,
                        rollout_method=method,
                        trial_config_path=trial_config_path,
                        **kwargs
                    )
                )
        # Gather the results as tasks complete.
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                all_results.append(result)
            except Exception as e:
                print("An error occurred during execution:", e)
    return all_results

def compile_results_to_dataframe(results):
    """
    Convert the list of metric dictionaries into a pandas DataFrame.
    """
    return pd.DataFrame(results)

def average_results(df, groupby_cols=['rollout_method', 'LI', 'EF', 'DF', 'hertg_method']):
    """
    Group the DataFrame by the specified parameters and compute the average of all numeric metrics.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing individual run results.
        groupby_cols (list of str): List of columns to group by.
    
    Returns:
        pd.DataFrame: DataFrame with averaged metrics.
    """
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    avg_df = df.groupby(groupby_cols)[numeric_cols].mean().reset_index()
    return avg_df


# Define the rollout methods you wish to test.
rollout_methods = ['random', 'same', 'random_same', 'zero', 'accelerate']

# Specify the number of trials.
# For example, if you have pre-saved configurations "trial_0.pkl", "trial_1.pkl", ..., "trial_9.pkl"
num_trials = 100

# Path to the directory where trial configurations are stored.
trial_config_path = "trial_configs1"  # Update this path as needed.

# Get list of pickle files in folder
trial_config_names = os.listdir(trial_config_path)
trial_config_names = [f for f in trial_config_names if f.endswith('.pkl')]
trial_config_names = [f[:-4] for f in trial_config_names] # Remove .pkl extension

# Run experiments for each rollout method over all the pre-generated configurations.
results = run_experiments_for_rollout_methods(
    rollout_methods,
    trial_config_path,
    trial_config_names,
    max_actions=200,
    LI=100,
    EF=0.1,
    DF=1.0,
    hertg_method='static',
    rollout_pre_collision_stop=True
)

# Compile individual run results into a DataFrame.
df = compile_results_to_dataframe(results)
print("Individual run metrics:")
print(df)

# # Compute average metrics grouped by the test parameters.
# avg_df = average_results(df)
# print("\nAveraged metrics:")
# print(avg_df)

# Optionally, save the results to CSV files.
df.to_csv("mcts_metrics_same_states_100trials.csv", index=False)
# avg_df.to_csv("mcts_metrics_averaged.csv", index=False)


Toy Measurement Control InitializedToy Measurement Control Initialized
Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized

Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized
Toy Measurement Control Initialized

Toy Measurement Control Initialized
Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized



/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


Toy Measurement Control Initialized


/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)




Toy Measurement Control Initialized



...

/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


Toy Measurement Control InitializedToy Measurement Control Initialized



/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)
/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 

In [9]:
# -----------------------------------------------------------------------------
# Worker wrapper: creates a fresh environment and runs one trial of the metrics.
# -----------------------------------------------------------------------------
def worker_wrapper(rollout_method,
                   max_actions=200,
                   LI=100,
                   EF=0.1,
                   DF=1.0,
                   HL=6,
                   hertg_method='static',
                   rollout_pre_collision_stop=True):
    """
    Creates a fresh environment, resets it to obtain a new random state, and
    runs the MCTS metrics collection with the given rollout method.
    """
    
    # Get a random starting state and the corresponding true object state.
    state = env.reset()
    object_true_state = env.object_manager.get_true_state()
    
    # Call your provided get_mcts_metrics function.
    metrics = get_mcts_metrics(state, object_true_state,
                               max_actions=max_actions,
                               LI=LI,
                               EF=EF,
                               DF=DF,
                               HL=HL,
                               rollout_method=rollout_method,
                               hertg_method=hertg_method,
                               rollout_pre_collision_stop=rollout_pre_collision_stop)
    return metrics

# -----------------------------------------------------------------------------
# Experiment runner: submits a series of tasks in parallel for each rollout method.
# -----------------------------------------------------------------------------
def run_experiments_for_rollout_methods(rollout_methods, num_trials=10, **kwargs):
    """
    For each rollout method in rollout_methods, run num_trials independent experiments.
    Additional parameters for get_mcts_metrics can be passed via kwargs.
    
    Returns:
        A list of dictionaries containing the metrics for each run.
    """
    all_results = []
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = []
        # For each rollout method, schedule num_trials experiments.
        for method in rollout_methods:
            for _ in range(num_trials):
                # Submit the task to the process pool.
                futures.append(executor.submit(worker_wrapper, method, **kwargs))
        
        # Collect the results as they complete.
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                all_results.append(result)
            except Exception as e:
                print("An error occurred during execution:", e)
    return all_results

# -----------------------------------------------------------------------------
# Results compilation: create a DataFrame from the list of result dictionaries.
# -----------------------------------------------------------------------------
def compile_results_to_dataframe(results):
    """
    Converts the list of metric dictionaries into a Pandas DataFrame.
    """
    return pd.DataFrame(results)

# -----------------------------------------------------------------------------
# Averaging results: group by fixed parameters and average numeric metrics.
# -----------------------------------------------------------------------------
def average_results(df, groupby_cols=['rollout_method', 'LI', 'EF', 'DF', 'hertg_method']):
    """
    Given a DataFrame of results, group by the parameter columns and compute the
    average of the numeric metric columns.
    """
    # Identify numeric columns to average.
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    avg_df = df.groupby(groupby_cols)[numeric_cols].mean().reset_index()
    return avg_df

# -----------------------------------------------------------------------------
# Main function: defines the experiment and saves/prints the results.
# -----------------------------------------------------------------------------
# Define the rollout methods you want to test.
rollout_methods = ['random', 'same', 'random_same', 'zero', 'accelerate']

# Set the number of trials per rollout method.
num_trials = 100  # Adjust as needed for better averaging.

# Run the experiments. You can pass additional parameters (like max_actions, etc.)
results = run_experiments_for_rollout_methods(rollout_methods, num_trials=num_trials)

# Compile the individual run results into a DataFrame.
df = compile_results_to_dataframe(results)
print("Individual run metrics:")
display(df)

# # Compute average metrics per parameter configuration.
# avg_df = average_results(df)
# print("\nAveraged metrics:")
# display(avg_df)

# Optionally, save the results to CSV files.
df.to_csv("mcts_metrics_individual_100_trials.csv", index=False)
# avg_df.to_csv("mcts_metrics_averaged.csv", index=False)

# -----------------------------------------------------------------------------
# Run the main function.
# -----------------------------------------------------------------------------
# if __name__ == '__main__':
#     main()


Toy Measurement Control Initialized
Toy Measurement Control Initialized
Toy Measurement Control Initialized
Toy Measurement Control Initialized

/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized
Toy Measurement Control Initialized

/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)
/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)






Toy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control InitializedToy Measurement Control Initialized
Toy Measurement Control Initialized
Toy Measurement Control Initialized
Toy Measurement Control InitializedToy Measurement Control Initialized

...

Toy Measurement Control InitializedToy Measurement Control Initialized


/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


/home/austin/MeasurementMCTS/metrics/../src/measurement_mcts/measurement_mcts/mcts/mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
sum_traces: 0.0
Toy Measurement Control Initialized
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.][0. 0. 0.

,LI,EF,DF,HL,rollout_method,hertg_method,done,percent_done,cumulative_reward,num_actions,computation_time,computation_per_action
0,100,0.1,1.0,6,random,static,True,100.000000,6.950000,72,132.636248,1.842170
1,100,0.1,1.0,6,random,static,True,100.000000,6.950000,72,133.341655,1.851967
2,100,0.1,1.0,6,random,static,True,100.000000,6.950000,72,135.203082,1.877821
3,100,0.1,1.0,6,random,static,True,100.000000,6.950000,72,136.183471,1.891437
4,100,0.1,1.0,6,random,static,True,100.000000,6.950000,72,136.478992,1.895542
...,...,...,...,...,...,...,...,...,...,...,...,...
495,100,0.1,1.0,6,accelerate,static,False,91.666667,6.351786,200,429.192076,2.145960
496,100,0.1,1.0,6,accelerate,static,False,91.666667,6.351786,200,436.171579,2.180858
497,100,0.1,1.0,6,accelerate,static,False,91.666667,6.370833,200,281.863650,1.409318
498,100,0.1,1.0,6,accelerate,static,False,91.666667,6.370833,200,235.196876,1.175984


In [7]:
df.to_csv("mcts_metrics_individual.csv", index=False)